# 03: ベクターデータベース (VDB) の作成とデプロイ

このノートブックでは、EC 市場調査 PDF をもとにベクターデータベースを構築し、RAG 用にデプロイします。

**参考**: agent-build-clinic の Notebook 5 (PDF Onboarding) と同じアプローチで、
DataRobot Python Client の `VectorDatabase.create()` を使用します。

## 処理の流れ
1. 環境変数の読み込みと DataRobot クライアントの初期化
2. PDF ドキュメントの準備 → ZIP → AI Catalog アップロード
3. VectorDatabase の作成 (Python Client)
4. VDB → カスタムモデル → 登録 → デプロイ
5. `.env` に設定する VDB_DEPLOYMENT_ID の出力

## 1. 環境変数の読み込み

In [ ]:
import os
import time
import json
import zipfile
import pathlib
import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv

load_dotenv("../.env", override=True)

DATAROBOT_API_TOKEN = os.environ["DATAROBOT_API_TOKEN"]
DATAROBOT_ENDPOINT = os.environ.get("DATAROBOT_ENDPOINT", "https://app.datarobot.com/api/v2")

print(f"Endpoint: {DATAROBOT_ENDPOINT}")
print(f"Token:    {DATAROBOT_API_TOKEN[:8]}...")

## 2. DataRobot クライアントの初期化

In [ ]:
import datarobot as dr
import requests

client = dr.Client(
    token=DATAROBOT_API_TOKEN,
    endpoint=DATAROBOT_ENDPOINT,
)
print(f"DataRobot client initialized (v{dr.__version__})")

# REST API 用のヘッダー
API_BASE = DATAROBOT_ENDPOINT
HEADERS = {
    "Authorization": f"Bearer {DATAROBOT_API_TOKEN}",
    "Content-Type": "application/json",
}

## 3. PDF ドキュメントの準備

経済産業省の EC 市場調査 PDF を `../documents/` ディレクトリに配置してください。

### 方法 A: 手動配置
`../documents/` フォルダに PDF ファイルを配置してください。

### 方法 B: ダウンロード (以下のセルを実行)
経産省の EC 市場調査レポートをダウンロードします。

In [ ]:
DOCUMENTS_DIR = pathlib.Path("../documents").resolve()
DOCUMENTS_DIR.mkdir(parents=True, exist_ok=True)

# 経産省 EC市場調査 PDF のURL (適宜更新してください)
# 例: 令和5年度 電子商取引に関する市場調査
PDF_URL = "https://www.meti.go.jp/press/2024/09/20240925001/20240925001-1.pdf"
PDF_FILENAME = "ec_market_survey.pdf"
PDF_PATH = DOCUMENTS_DIR / PDF_FILENAME

if PDF_PATH.exists():
    print(f"PDF は既に存在します: {PDF_PATH}")
else:
    print(f"PDF をダウンロード中: {PDF_URL}")
    try:
        resp = requests.get(PDF_URL, timeout=120)
        resp.raise_for_status()
        PDF_PATH.write_bytes(resp.content)
        print(f"ダウンロード完了: {PDF_PATH} ({len(resp.content) / 1024 / 1024:.1f} MB)")
    except Exception as e:
        print(f"ダウンロードに失敗しました: {e}")
        print(f"手動で PDF を {DOCUMENTS_DIR} に配置してください。")

In [ ]:
# documents ディレクトリ内の PDF を確認
pdf_files = list(DOCUMENTS_DIR.glob("*.pdf"))
if not pdf_files:
    raise FileNotFoundError(
        f"PDF ファイルが見つかりません。{DOCUMENTS_DIR} にPDFを配置してください。"
    )

print(f"使用する PDF ファイル:")
for f in pdf_files:
    print(f"  - {f.name} ({f.stat().st_size / 1024 / 1024:.1f} MB)")

## 4. PDF を ZIP に圧縮して AI Catalog にアップロード

DataRobot の VectorDatabase はZIPファイルからドキュメントを読み込みます。

In [ ]:
ZIP_PATH = DOCUMENTS_DIR / "ec_documents.zip"

with zipfile.ZipFile(str(ZIP_PATH), "w", zipfile.ZIP_DEFLATED) as zf:
    for pdf_file in pdf_files:
        zf.write(str(pdf_file), pdf_file.name)
        print(f"  ZIP に追加: {pdf_file.name}")

print(f"\nZIP ファイルを作成しました: {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1024 / 1024:.1f} MB)")

In [ ]:
# AI Catalog にアップロード
CATALOG_NAME = "ec_market_survey_documents"

catalog_datasets = dr.Dataset.list()
existing = [d for d in catalog_datasets if d.name == CATALOG_NAME]

if existing:
    doc_dataset = existing[0]
    print(f"既存のカタログアイテムを使用します: {doc_dataset.name} (ID: {doc_dataset.id})")
else:
    doc_dataset = dr.Dataset.create_from_file(file_path=str(ZIP_PATH))
    print(f"カタログにアップロードしました: {doc_dataset.name} (ID: {doc_dataset.id})")

print(f"\nDataset ID: {doc_dataset.id}")

## 3. VectorDatabase の作成 (Python Client)

agent-build-clinic と同じ `VectorDatabase.create()` を使用します。

| パラメータ | 値 |
|---|---|
| Embedding モデル | JINA_EMBEDDING_T_EN_V1 |
| チャンキング方式 | RECURSIVE |
| チャンクサイズ | 256 |
| オーバーラップ率 | 25% |

In [ ]:
# Use Case を取得
existing_use_case_id = os.environ.get("DATAROBOT_DEFAULT_USE_CASE", "").strip()
if existing_use_case_id:
    use_case = dr.UseCase.get(existing_use_case_id)
else:
    # 前のノートブックで作成した Use Case を名前で検索
    use_cases = dr.UseCase.list()
    matching = [uc for uc in use_cases if "小売EC需要予測" in uc.name]
    if matching:
        use_case = matching[0]
    else:
        use_case = dr.UseCase.create(name="小売EC需要予測エージェント")

print(f"Use Case: {use_case.name} (ID: {use_case.id})")

In [ ]:
# VectorDatabase 作成 (Python Client - agent-build-clinic 方式)
from datarobot.models.genai.vector_database import VectorDatabase, ChunkingParameters
from datarobot.enums import VectorDatabaseEmbeddingModel, VectorDatabaseChunkingMethod

chunking_parameters = ChunkingParameters(
    embedding_model=VectorDatabaseEmbeddingModel.JINA_EMBEDDING_T_EN_V1,
    chunking_method=VectorDatabaseChunkingMethod.RECURSIVE,
    chunk_size=256,
    chunk_overlap_percentage=25,
    separators=["\n\n", "\n", " ", ""],
)

print("VectorDatabase を作成中...")
print(f"  Embedding: JINA_EMBEDDING_T_EN_V1")
print(f"  Chunking: RECURSIVE, size=256, overlap=25%")

vdb = VectorDatabase.create(
    dataset_id=doc_dataset.id,
    chunking_parameters=chunking_parameters,
    use_case=use_case.id,
    name="EC市場調査_VectorDatabase",
)

VDB_ID = vdb.id
print(f"\n✅ VectorDatabase を作成しました。")
print(f"  VDB ID: {VDB_ID}")

## 4. VDB 構築完了の待機 → デプロイ

In [ ]:
# VDB 構築完了の待機
print("VDB のインデックス構築完了を待機しています...")
print(f"  (通常 2〜5 分程度かかります)")

start = time.time()
while True:
    vdb_refreshed = VectorDatabase.get(VDB_ID)
    status = vdb_refreshed.execution_status
    elapsed = time.time() - start
    print(f"  Status: {status} (elapsed: {elapsed:.0f}s)")
    
    if status == "COMPLETED":
        print(f"\n✅ VDB 構築が完了しました。")
        break
    elif status in ("ERROR", "FAILED"):
        raise RuntimeError(f"VDB 構築に失敗しました: {status}")
    
    if elapsed > 600:
        raise TimeoutError("VDB 構築が10分以内に完了しませんでした。")
    
    time.sleep(15)

## 5. カスタムモデル → 登録 → デプロイ

agent-build-clinic (Notebook 5) と同じフローで VDB をデプロイメントにします。

In [ ]:
# ===== Step 1: カスタムモデルワークショップへ送信 =====
from datarobot.models.genai.vector_database import CustomModelValidation

print("カスタムモデルワークショップへ送信中...")
validation = vdb_refreshed.send_to_custom_model_workshop()

# validation からカスタムモデル情報を取得
if hasattr(validation, 'custom_model_version_id'):
    CUSTOM_MODEL_VERSION_ID = validation.custom_model_version_id
    CUSTOM_MODEL_ID = validation.custom_model_id
elif isinstance(validation, dict):
    CUSTOM_MODEL_VERSION_ID = validation.get("customModelVersionId")
    CUSTOM_MODEL_ID = validation.get("customModelId")
else:
    # validation オブジェクトの属性を確認
    print(f"  validation type: {type(validation)}")
    print(f"  validation attrs: {[a for a in dir(validation) if not a.startswith('_')]}")
    CUSTOM_MODEL_VERSION_ID = getattr(validation, 'custom_model_version_id', None)
    CUSTOM_MODEL_ID = getattr(validation, 'custom_model_id', None)

print(f"  Custom Model ID: {CUSTOM_MODEL_ID}")
print(f"  Custom Model Version ID: {CUSTOM_MODEL_VERSION_ID}")

## ビルド待機 → 登録 → デプロイ

In [ ]:
# ===== Step 2: ビルド完了の待機 =====
print("カスタムモデルのビルド完了を待機しています...")
print(f"  (通常 3〜8 分程度かかります)")

from datarobot.models.custom_model_version import CustomModelVersion

start = time.time()
while True:
    try:
        cmv = CustomModelVersion.get(CUSTOM_MODEL_ID, CUSTOM_MODEL_VERSION_ID)
        build_status = cmv.build_status if hasattr(cmv, 'build_status') else "checking..."
        elapsed = time.time() - start
        print(f"  Build Status: {build_status} (elapsed: {elapsed:.0f}s)")
        
        if build_status == "success":
            print(f"\n✅ ビルドが完了しました。")
            break
        elif build_status in ("failed", "error"):
            raise RuntimeError(f"ビルドに失敗しました: {build_status}")
    except Exception as e:
        if "success" in str(e).lower() or "not found" in str(e).lower():
            print(f"  例外 (リトライ): {e}")
        else:
            raise
    
    if time.time() - start > 600:
        raise TimeoutError("ビルドが10分以内に完了しませんでした。")
    
    time.sleep(15)

## 登録とデプロイ

In [ ]:
# ===== Step 3: 登録 → デプロイ =====
from datarobot.models.model_registry import RegisteredModel

# 登録
print("モデルパッケージとして登録中...")
registered = RegisteredModel.create_for_custom_model_version(
    custom_model_version_id=CUSTOM_MODEL_VERSION_ID,
    registered_model_name="EC市場調査_VectorDatabase",
)
versions = registered.list_versions()
latest_version = versions[0]
print(f"  Registered Model Version ID: {latest_version.id}")

# デプロイ
print("\nデプロイ中...")
prediction_environment = dr.PredictionEnvironment.list()[0]
print(f"  予測環境: {prediction_environment.name}")

deployment = dr.Deployment.create(
    model_package_id=latest_version.id,
    label="EC市場調査_VDB_デプロイメント",
    default_prediction_server_id=prediction_environment.id,
    use_case_id=use_case.id,
)

VDB_DEPLOYMENT_ID = deployment.id
print(f"\n✅ デプロイ完了!")
print(f"  VDB Deployment ID: {VDB_DEPLOYMENT_ID}")

## 結果出力

In [ ]:
print("=" * 60)
print("以下を .env ファイルに追記してください:")
print("=" * 60)
print(f"VDB_DEPLOYMENT_ID={VDB_DEPLOYMENT_ID}")
print("=" * 60)
print(f"\nVectorDatabase ID: {VDB_ID}")
print(f"Use Case ID:       {use_case.id}")
print("\n完了: 次のステップは 04_setup_prompt_and_mcp.ipynb に進んでください。")